# Step 1 — Collect Newly Registered Accounts

By replicating Zheng et al. (2019), the following code fetches all accounts registered between March–May 2025 directly from Wikipedia's new user log (Special:Log/newusers).

Output: `data/new_users.jsonl` — one registration event per line.
Next step: filter to users who made at least one edit (notebook 02).

In [1]:
import json
import time
import requests
from pathlib import Path

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────

API_URL       = "https://en.wikipedia.org/w/api.php"
USER_AGENT    = "BDACA-II-course/1.0 (educational; adashiyj@gmail.com)"
REQUEST_DELAY = 0.5
TIMEOUT       = 30

# Sample newcomers by registration date directly (Zheng et al., 2019 approach)
# No specific articles needed — avoids semi-protection bias
#WINDOW_START = "2025-03-01T00:00:00Z"
#WINDOW_END   = "2025-05-31T23:59:59Z"

# for testing ONLY
WINDOW_START = "2026-03-10T00:00:00Z"
WINDOW_END   = "2026-03-10T23:59:59Z"

OUTPUT_FILE = Path("data/new_users.jsonl")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

In [3]:
# ── Session setup  ──────────────────────────────────

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})


def api_get(params: dict) -> dict | None:
    """Make a safe GET request to the MediaWiki API with error handling."""
    try:
        response = session.get(API_URL, params=params, timeout=TIMEOUT)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError:
        print("Response was not valid JSON")
        return None
    finally:
        time.sleep(REQUEST_DELAY)
    return data

In [4]:
# ── Fetch newly registered accounts in the window ─────────────────────────────

def get_new_users() -> list[dict]:
    """Fetch all accounts registered between WINDOW_START and WINDOW_END."""
    users = []
    params = {
        "action":  "query",
        "format":  "json",
        "list":    "logevents",
        "letype":  "newusers",
        "ledir":   "newer",
        "lestart": WINDOW_START,
        "leend":   WINDOW_END,
        "lelimit": "500",
    }

    while True:
        data = api_get(params)
        if data is None:
            print("Failed to fetch new users — stopping")
            break

        events = data.get("query", {}).get("logevents", [])
        users.extend(events)
        print(f"  Fetched so far: {len(users)} users", end="\r")

        if "continue" in data:
            params["lecontinue"] = data["continue"]["lecontinue"]
        else:
            break

    print(f"\nTotal new accounts fetched: {len(users)}")
    return users

In [5]:
# ── Run ────────────────────────────────────────────────────────────────────────

# Delete data/new_users.jsonl first if re-running from scratch
if OUTPUT_FILE.exists():
    print(f"File already exists — loading from checkpoint")
    with OUTPUT_FILE.open() as f:
        users = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(users)} users from checkpoint")
else:
    users = get_new_users()
    with OUTPUT_FILE.open("w", encoding="utf-8") as f:
        for u in users:
            f.write(json.dumps(u, ensure_ascii=False) + "\n")

print(f"Saved to: {OUTPUT_FILE}")

  Fetched so far: 11546 users
Total new accounts fetched: 11546
Saved to: data/new_users.jsonl


In [6]:
# ── Quick check ────────────────────────────────────────────────────────────────

import pandas as pd

df = pd.read_json(OUTPUT_FILE, lines=True)
print(f"Total registered accounts: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample rows:")
df[["user", "timestamp", "type"]].head(10)

Total registered accounts: 11546

Columns: ['logid', 'ns', 'title', 'pageid', 'logpage', 'params', 'type', 'action', 'user', 'temp', 'timestamp', 'comment', 'actionhidden', 'userhidden', 'suppressed']

Sample rows:


,user,timestamp,type
0,~2026-15124-48,2026-03-10 00:00:04+00:00,newusers
1,~2026-13541-00,2026-03-10 00:00:07+00:00,newusers
2,Good 67MangoMustard69,2026-03-10 00:00:08+00:00,newusers
3,~2026-15111-04,2026-03-10 00:00:15+00:00,newusers
4,~2026-13671-36,2026-03-10 00:00:25+00:00,newusers
5,Idk123999,2026-03-10 00:00:38+00:00,newusers
6,~2026-2321,2026-03-10 00:00:46+00:00,newusers
7,~2026-14972-58,2026-03-10 00:00:54+00:00,newusers
8,~2026-15089-85,2026-03-10 00:00:56+00:00,newusers
9,~2026-15190-72,2026-03-10 00:00:57+00:00,newusers
